In [1]:
from dj_notebook import activate

plus = activate()

Output()

In [2]:
import pandas as pd
from django.db import transaction
from cuentas.models import User
from empresas.models import SucursalEmpresa, UsuarioEmpresa, Empresa


In [22]:
from ordentrabajo.models import (
    OrdenDeTrabajo, UsuarioAsignadoOT, DetalleTrabajo, 
    SeguimientoDetalleTrabajo, HistorialCambiosOrden, 
    AdjuntoDeOrden, DetalleGastoRendicionOT
)

def analizar_orden_trabajo(orden_id):
    """
    Extrae y muestra información detallada de una Orden de Trabajo
    
    Args:
        orden_id: ID de la orden de trabajo a analizar
    
    Returns:
        dict: Diccionario con toda la información de la OT
    """
    try:
        orden = OrdenDeTrabajo.objects.select_related(
            'empresa', 'cliente', 'responsable_empresa', 'solicitante_empresa',
            'responsable_empresa__usuario', 'solicitante_empresa__usuario'
        ).prefetch_related(
            'usuarioasignadoot_set__usuario_empresa__usuario',
            'detalletrabajo_set__tecnico_asignado__usuario',
            'historial__usuario__usuario',  # related_name="historial"
            'adjuntodeorden_set',
            'detallegastorendicionot_set__categoria'
        ).get(id=orden_id)
        
        # Información básica de la orden
        info = {
            'id': orden.id,
            'empresa': {
                'id': orden.empresa.id,
                'nombre': orden.empresa.nombre,
                'rut': orden.empresa.rut_empresa
            },
            'cliente': {
                'id': orden.cliente.id,
                'nombre': orden.cliente.nombre,
                'rut': orden.cliente.rut_empresa
            },
            'fechas': {
                'creacion': orden.fecha_creacion,
                'modificacion': orden.fecha_modificacion,
                'inicio_ot': orden.fecha_inicio_ot,
                'finalizacion_ot': orden.fecha_finalizacion_ot
            },
            'estado': orden.estado,
            'prioridad': orden.prioridad,
            'descripcion': orden.descripcion,
            'notas_internas': orden.notas_internas,
            'responsable': None,
            'solicitante': None,
            'usuarios_asignados': [],
            'detalles_trabajo': [],
            'historial_cambios': [],
            'adjuntos': [],
            'gastos_rendicion': []
        }
        
        # Responsable
        if orden.responsable_empresa:
            info['responsable'] = {
                'id': orden.responsable_empresa.id,
                'usuario_id': orden.responsable_empresa.usuario.id,
                'nombre': orden.responsable_empresa.usuario.get_nombre(),
                'email': orden.responsable_empresa.usuario.email
            }
        
        # Solicitante
        if orden.solicitante_empresa:
            info['solicitante'] = {
                'id': orden.solicitante_empresa.id,
                'usuario_id': orden.solicitante_empresa.usuario.id,
                'nombre': orden.solicitante_empresa.usuario.get_nombre(),
                'email': orden.solicitante_empresa.usuario.email
            }
        
        # Usuarios asignados
        for asignado in orden.usuarioasignadoot_set.all():
            usuario_info = {
                'id': asignado.id,
                'fecha_creacion': asignado.fecha_creacion
            }
            if asignado.usuario_empresa:
                usuario_info['tipo'] = 'interno'
                usuario_info['nombre'] = asignado.usuario_empresa.usuario.get_nombre()
                usuario_info['email'] = asignado.usuario_empresa.usuario.email
            else:
                usuario_info['tipo'] = 'externo'
                usuario_info['nombre'] = asignado.usuario_externo
                usuario_info['email'] = asignado.correo_usuario_externo
            info['usuarios_asignados'].append(usuario_info)
        
        # Detalles de trabajo
        for detalle in orden.detalletrabajo_set.all():
            detalle_info = {
                'id': detalle.id,
                'nombre': detalle.nombre,
                'descripcion': detalle.descripcion,
                'estado': detalle.estado,
                'fecha_creacion': detalle.fecha_creacion,
                'tecnico_asignado': None,
                'trabajo_relacionado': None,
                'insumo': None,
                'seguimientos': []
            }
            
            # Técnico asignado
            if detalle.tecnico_asignado:
                detalle_info['tecnico_asignado'] = {
                    'id': detalle.tecnico_asignado.id,
                    'nombre': detalle.tecnico_asignado.usuario.get_nombre(),
                    'email': detalle.tecnico_asignado.usuario.email
                }
            
            # Trabajo relacionado (GenericForeignKey)
            if detalle.trabajo:
                detalle_info['trabajo_relacionado'] = {
                    'tipo': detalle.content_type.model,
                    'id': detalle.trabajo_id
                }
            
            # Insumo (GuiaSalida)
            if detalle.insumo:
                detalle_info['insumo'] = {
                    'id': detalle.insumo.id,
                    'numero': getattr(detalle.insumo, 'numero', None)
                }
            
            # Seguimientos del detalle - cargamos sin prefetch para evitar conflictos
            seguimientos = SeguimientoDetalleTrabajo.objects.filter(
                detalle_trabajo=detalle
            ).select_related('usuario__usuario')
            
            for seguimiento in seguimientos:
                seguimiento_info = {
                    'id': seguimiento.id,
                    'tipo': seguimiento.tipo,
                    'fecha': seguimiento.fecha,
                    'comentario': seguimiento.comentario,
                    'usuario': None
                }
                if seguimiento.usuario:
                    seguimiento_info['usuario'] = {
                        'id': seguimiento.usuario.id,
                        'nombre': seguimiento.usuario.usuario.get_nombre()
                    }
                detalle_info['seguimientos'].append(seguimiento_info)
            
            info['detalles_trabajo'].append(detalle_info)
        
        # Historial de cambios - usar related_name="historial"
        for cambio in orden.historial.all():
            cambio_info = {
                'id': cambio.id,
                'fecha_cambio': cambio.fecha_cambio,
                'estado_anterior': cambio.estado_anterior,
                'estado_actual': cambio.estado_actual,
                'comentario': cambio.comentario,
                'usuario': {
                    'id': cambio.usuario.id,
                    'nombre': cambio.usuario.usuario.get_nombre()
                }
            }
            info['historial_cambios'].append(cambio_info)
        
        # Adjuntos
        for adjunto in orden.adjuntodeorden_set.all():
            adjunto_info = {
                'id': adjunto.id,
                'tipo': adjunto.tipo,
                'descripcion': adjunto.descripcion,
                'archivo': adjunto.archivo.name if adjunto.archivo else None,
                'fecha_creacion': adjunto.fecha_creacion
            }
            info['adjuntos'].append(adjunto_info)
        
        # Gastos de rendición
        for gasto in orden.detallegastorendicionot_set.all():
            gasto_info = {
                'id': gasto.id,
                'categoria': {
                    'id': gasto.categoria.id,
                    'nombre': gasto.categoria.nombre
                },
                'detalle': gasto.detalle,
                'cantidad': gasto.cantidad,
                'monto_unitario': gasto.monto_unitario,
                'monto_total': gasto.monto_total,
                'fecha_gasto': gasto.fecha_gasto
            }
            info['gastos_rendicion'].append(gasto_info)
        
        # Resumen de estadísticas
        info['estadisticas'] = {
            'total_usuarios_asignados': len(info['usuarios_asignados']),
            'total_detalles_trabajo': len(info['detalles_trabajo']),
            'total_seguimientos': sum(len(d['seguimientos']) for d in info['detalles_trabajo']),
            'total_cambios': len(info['historial_cambios']),
            'total_adjuntos': len(info['adjuntos']),
            'total_gastos': len(info['gastos_rendicion']),
            'suma_gastos': sum(g['monto_total'] for g in info['gastos_rendicion'])
        }
        
        return info
        
    except OrdenDeTrabajo.DoesNotExist:
        print(f"❌ No existe una Orden de Trabajo con ID {orden_id}")
        return None
    except Exception as e:
        print(f"❌ Error al analizar la orden: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


def mostrar_resumen_ot(orden_id):
    """Muestra un resumen visual de la Orden de Trabajo"""
    info = analizar_orden_trabajo(orden_id)
    
    if not info:
        return
    
    print("=" * 80)
    print(f"📋 ORDEN DE TRABAJO #{info['id']}")
    print("=" * 80)
    print(f"\n🏢 EMPRESA: {info['empresa']['nombre']} (RUT: {info['empresa']['rut']})")
    print(f"👤 CLIENTE: {info['cliente']['nombre']} (RUT: {info['cliente']['rut']})")
    print(f"\n📊 ESTADO: {info['estado'].upper()}")
    print(f"⚡ PRIORIDAD: {info['prioridad']}")
    print(f"\n📝 DESCRIPCIÓN: {info['descripcion']}")
    
    if info['notas_internas']:
        print(f"\n🔒 NOTAS INTERNAS: {info['notas_internas']}")
    
    print(f"\n📅 FECHAS:")
    print(f"   • Creación: {info['fechas']['creacion']}")
    print(f"   • Inicio OT: {info['fechas']['inicio_ot'] or 'No definida'}")
    print(f"   • Finalización OT: {info['fechas']['finalizacion_ot'] or 'No definida'}")
    
    if info['responsable']:
        print(f"\n👔 RESPONSABLE: {info['responsable']['nombre']} ({info['responsable']['email']})")
    
    if info['solicitante']:
        print(f"🙋 SOLICITANTE: {info['solicitante']['nombre']} ({info['solicitante']['email']})")
    
    if info['usuarios_asignados']:
        print(f"\n👥 USUARIOS ASIGNADOS ({len(info['usuarios_asignados'])}):")
        for ua in info['usuarios_asignados']:
            tipo_icon = "🔧" if ua['tipo'] == 'interno' else "🌐"
            print(f"   {tipo_icon} {ua['nombre']} ({ua['email']})")
    
    if info['detalles_trabajo']:
        print(f"\n🔨 DETALLES DE TRABAJO ({len(info['detalles_trabajo'])}):")
        for dt in info['detalles_trabajo']:
            print(f"   • #{dt['id']}: {dt['nombre']} - Estado: {dt['estado']}")
            if dt['tecnico_asignado']:
                print(f"     Técnico: {dt['tecnico_asignado']['nombre']}")
            print(f"     Seguimientos: {len(dt['seguimientos'])}")
    
    if info['historial_cambios']:
        print(f"\n📜 HISTORIAL DE CAMBIOS ({len(info['historial_cambios'])}):")
        for cambio in info['historial_cambios'][:5]:  # Primeros 5
            print(f"   • {cambio['fecha_cambio']}: {cambio['usuario']['nombre']}")
            if cambio['estado_anterior']:
                print(f"     Antes: {cambio['estado_anterior'][:50]}...")
            if cambio['estado_actual']:
                print(f"     Después: {cambio['estado_actual'][:50]}...")
    
    if info['adjuntos']:
        print(f"\n📎 ADJUNTOS ({len(info['adjuntos'])}):")
        for adj in info['adjuntos']:
            print(f"   • {adj['tipo']}: {adj['descripcion'] or adj['archivo']}")
    
    if info['gastos_rendicion']:
        print(f"\n💰 GASTOS DE RENDICIÓN ({len(info['gastos_rendicion'])}):")
        for gasto in info['gastos_rendicion']:
            print(f"   • {gasto['categoria']['nombre']}: ${gasto['monto_total']:,}")
    
    print(f"\n📊 ESTADÍSTICAS:")
    print(f"   • Total Usuarios Asignados: {info['estadisticas']['total_usuarios_asignados']}")
    print(f"   • Total Detalles de Trabajo: {info['estadisticas']['total_detalles_trabajo']}")
    print(f"   • Total Seguimientos: {info['estadisticas']['total_seguimientos']}")
    print(f"   • Total Cambios Registrados: {info['estadisticas']['total_cambios']}")
    print(f"   • Total Adjuntos: {info['estadisticas']['total_adjuntos']}")
    print(f"   • Total Gastos: ${info['estadisticas']['suma_gastos']:,}")
    print("=" * 80)
    
    return info


# Ejemplo de uso:
# info = analizar_orden_trabajo(1)  # Devuelve diccionario completo
# mostrar_resumen_ot(1)  # Muestra resumen visual
print("✅ Funciones de análisis de OT cargadas:")
print("   • analizar_orden_trabajo(orden_id) - Extrae información completa en diccionario")
print("   • mostrar_resumen_ot(orden_id) - Muestra resumen visual en consola")

✅ Funciones de análisis de OT cargadas:
   • analizar_orden_trabajo(orden_id) - Extrae información completa en diccionario
   • mostrar_resumen_ot(orden_id) - Muestra resumen visual en consola


In [23]:
info = analizar_orden_trabajo(1)

In [27]:
# Mostrar resumen visual de la OT
mostrar_resumen_ot(1)

📋 ORDEN DE TRABAJO #1

🏢 EMPRESA: Snabbit (RUT: 11111111-1)
👤 CLIENTE: AYG ASOCIADOS (RUT: None)

📊 ESTADO: PENDIENTE
⚡ PRIORIDAD: 1

📝 DESCRIPCIÓN: Prueba OT1

📅 FECHAS:
   • Creación: 2025-11-07 19:56:45.377129+00:00
   • Inicio OT: 2025-11-07
   • Finalización OT: 2025-12-07

👔 RESPONSABLE: Juan Técnico (tecnico@snabbit.cl)
🙋 SOLICITANTE: Nathaly Aguilera (naguileran@aygasociados.cl)

👥 USUARIOS ASIGNADOS (1):
   🔧 Nathaly Aguilera (naguileran@aygasociados.cl)

🔨 DETALLES DE TRABAJO (1):
   • #1: Prueba Trabajo1 - Estado: pendiente
     Técnico: Juan Técnico
     Seguimientos: 1

📊 ESTADÍSTICAS:
   • Total Usuarios Asignados: 1
   • Total Detalles de Trabajo: 1
   • Total Seguimientos: 1
   • Total Cambios Registrados: 0
   • Total Adjuntos: 0
   • Total Gastos: $0


{'id': 1,
 'empresa': {'id': 1, 'nombre': 'Snabbit', 'rut': '11111111-1'},
 'cliente': {'id': 4, 'nombre': 'AYG ASOCIADOS', 'rut': None},
 'fechas': {'creacion': datetime.datetime(2025, 11, 7, 19, 56, 45, 377129, tzinfo=datetime.timezone.utc),
  'modificacion': datetime.datetime(2025, 11, 7, 19, 56, 45, 377129, tzinfo=datetime.timezone.utc),
  'inicio_ot': datetime.date(2025, 11, 7),
  'finalizacion_ot': datetime.date(2025, 12, 7)},
 'estado': 'pendiente',
 'prioridad': '1',
 'descripcion': 'Prueba OT1',
 'notas_internas': '',
 'responsable': {'id': 2,
  'usuario_id': 2,
  'nombre': 'Juan Técnico',
  'email': 'tecnico@snabbit.cl'},
 'solicitante': {'id': 5,
  'usuario_id': 5,
  'nombre': 'Nathaly Aguilera',
  'email': 'naguileran@aygasociados.cl'},
 'usuarios_asignados': [{'id': 1,
   'fecha_creacion': datetime.datetime(2025, 11, 7, 19, 56, 45, 405324, tzinfo=datetime.timezone.utc),
   'tipo': 'interno',
   'nombre': 'Nathaly Aguilera',
   'email': 'naguileran@aygasociados.cl'}],
 'det